<div style="background: linear-gradient(135deg, #1a3a5c 0%, #2d6a9f 100%); padding: 40px 32px 32px 32px; border-radius: 12px; margin-bottom: 8px;">
  <h1 style="color: #ffffff; font-size: 2.0em; margin: 0 0 6px 0; font-family: 'Segoe UI', sans-serif; font-weight: 700;">
    Week 6, Lesson 11 -- Unsupervised Learning
  </h1>
  <h2 style="color: #a8d4f5; font-size: 1.2em; margin: 0 0 18px 0; font-family: 'Segoe UI', sans-serif; font-weight: 400;">
    Clustering and PCA for Reservoir Zonation
  </h2>
  <hr style="border: 1px solid rgba(255,255,255,0.25); margin: 16px 0;">
  <table style="color: #cce4ff; font-family: 'Segoe UI', sans-serif; font-size: 0.95em;">
    <tr>
      <td style="padding: 3px 24px 3px 0;"><strong>Week:</strong></td><td>6 (Lesson 11) -- Unsupervised Learning</td>
      <td style="padding: 3px 24px 3px 32px;"><strong>Duration:</strong></td><td>45 Minutes</td>
    </tr>
    <tr>
      <td style="padding: 3px 24px 3px 0;"><strong>Track:</strong></td><td>Petroleum Engineers &amp; Geoscientists</td>
      <td style="padding: 3px 24px 3px 32px;"><strong>Audience:</strong></td><td>Engineers &amp; Geoscientists</td>
    </tr>
    <tr>
      <td style="padding: 3px 24px 3px 0;"><strong>Instructor:</strong></td><td>Dr. Daniel Wamriew</td>
      <td style="padding: 3px 24px 3px 32px;"><strong>Contact:</strong></td><td>wamriewdan@gmail.com</td>
    </tr>
  </table>
</div>


<div style="background: #e8f5e9; border-left: 5px solid #2ca87f; padding: 14px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

### How to Use This Notebook

Work through this notebook **from top to bottom**, pressing **Shift + Enter** on each cell to run it.

- **Code cells** contain Python -- run them and read the output carefully.
- **Markdown cells** (white background) contain explanations -- read before running the next cell.
- **Student Activity** cells are marked with `Activity` -- complete these before moving on.
- **Homework** cells are marked with `Homework` -- complete after the session.
- You need **`well_log_data.csv`** in the course `data` folder.
- Install scikit-learn once if needed: **`pip install scikit-learn`**

</div>


---
## Table of Contents

1. [Why Unsupervised Learning?](#1-why-unsupervised-learning)
2. [Supervised vs Unsupervised, Revisited](#2-supervised-vs-unsupervised-revisited)
3. [What is K-Means Clustering?](#3-what-is-k-means-clustering)
4. [What is PCA?](#4-what-is-pca)
5. [Prepare the Data](#5-prepare-the-data)
6. [Feature Scaling](#6-feature-scaling)
7. [Choosing K: The Elbow Method](#7-choosing-k-the-elbow-method)
8. [Train K-Means and Inspect Clusters](#8-train-k-means-and-inspect-clusters)
9. [Compare Clusters to Known Formations](#9-compare-clusters-to-known-formations)
10. [Dimensionality Reduction with PCA](#10-dimensionality-reduction-with-pca)
11. [Visualizing Clusters in PCA Space](#11-visualizing-clusters-in-pca-space)
12. [Student Activity](#12-student-activity)
13. [Recap & Homework](#13-recap-homework)


---
## 1. Why Unsupervised Learning?

<div style="background: #f0f7ff; border-left: 5px solid #2d6a9f; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

In Lessons 9 and 10 you trained models to predict a **known label**: `Formation`.
That label existed because someone had already interpreted the well logs -- from core data, biostratigraphy, or expert picks.

In real fields, that is not always true. Sometimes you have:

- A **new well** with no formation tops yet picked
- A **new field** with no core data
- Thousands of depth samples and no time for a geologist to label every one by hand

**Unsupervised learning** finds structure in the data **without being told the answer**.
Instead of predicting a label, it groups similar rows together based on how alike their log responses are.

This is exactly what a geoscientist does informally when scanning a log track for "zones that look similar" --
unsupervised learning automates and quantifies that process.

</div>


---
## 2. Supervised vs Unsupervised, Revisited

<div style="background: #f0f7ff; border-left: 5px solid #2d6a9f; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

| | Supervised (Lessons 9-10) | Unsupervised (This Lesson) |
|---|---|---|
| **Target `y`** | Known (`Formation`) | None |
| **Goal** | Predict the correct label | Discover groups or structure |
| **Example algorithms** | Decision Tree, Random Forest | K-Means, PCA, hierarchical clustering |
| **Output** | A predicted class | A cluster number (0, 1, 2, ...) with no inherent meaning |
| **Petroleum example** | Classify formation from logs | Zone a reservoir with no prior labels |

A key mindset shift: a cluster labeled `2` is **not** automatically "Reservoir_2".
Clusters are numbered arbitrarily -- the engineer or geoscientist must interpret what each cluster represents afterward.

</div>


---
## 3. What is K-Means Clustering?

<div style="background: #f0f7ff; border-left: 5px solid #2d6a9f; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

**K-Means** groups rows into **k** clusters using only the feature values -- no target needed.

The algorithm, in plain terms:

```
1. Pick k starting points (centroids), e.g. k = 5
2. Assign every row to its nearest centroid
3. Move each centroid to the average position of the rows assigned to it
4. Repeat steps 2-3 until the centroids stop moving
```

Each depth sample ends up assigned to the cluster whose "average log response" it most resembles.

**Why this matters for reservoir zonation:**

- Rows with similar GR, NPHI, RHOB, RT, and SW values naturally end up in the same cluster
- Those groupings often correspond to real geological zones, even without formation labels
- It gives a fast, repeatable first pass at zoning a well before detailed picks are made

</div>


---
## 4. What is PCA?

<div style="background: #f0f7ff; border-left: 5px solid #2d6a9f; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

We have **5 log measurements** per row. Humans cannot easily visualize 5 dimensions at once.

**Principal Component Analysis (PCA)** compresses many correlated features into a small number of new axes
(called **principal components**) that capture as much of the original variation as possible.

```
5 correlated log measurements  -->  PCA  -->  2 new axes (PC1, PC2)
GR, NPHI, RHOB, RT, SW                        capture most of the spread in the data
```

PCA does **not** know about clusters or formations -- it only looks at how the features vary and covary.
We use it here purely to **visualize** our 5-dimensional data in a 2D scatter plot.

</div>


---
## 5. Prepare the Data

We use the same `well_log_data.csv` and the same five log features from Lessons 9 and 10.
Using the same features lets us compare the unsupervised clusters directly against the `Formation` labels we already know.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

print("Libraries loaded.")


In [ ]:
# Load and clean data
df = pd.read_csv("../data/well_log_data.csv")

log_cols = ["GR_API", "NPHI_frac", "RHOB_gcc", "RT_ohmm", "SW_frac"]
df[log_cols] = df.groupby("Well_ID")[log_cols].ffill()
df[log_cols] = df.groupby("Well_ID")[log_cols].bfill()

feature_cols = ["GR_API", "NPHI_frac", "RHOB_gcc", "RT_ohmm", "SW_frac"]
X = df[feature_cols].copy()

print("Shape:", df.shape)
print("Wells:", list(df["Well_ID"].unique()))
print("Missing log values remaining:", df[log_cols].isna().sum().sum())
print("\nFormation counts (known labels, used only for later comparison):")
print(df["Formation"].value_counts())

df.head()


---
## 6. Feature Scaling

<div style="background: #fff8e1; border-left: 5px solid #e0a800; padding: 14px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

Decision Trees and Random Forests do not care about the scale of a feature -- they only ask "is this value above or below a threshold?"

**K-Means and PCA are different.** Both rely on **distance** between rows.
`RT_ohmm` can range into the hundreds, while `NPHI_frac` and `SW_frac` sit between 0 and 1.
Without scaling, `RT_ohmm` would dominate the distance calculation simply because its numbers are bigger --
not because it is more geologically important.

**`StandardScaler`** puts every feature on the same footing: mean = 0, standard deviation = 1.

</div>


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Before scaling (mean, std):")
print(X.agg(["mean", "std"]).T)

print("\nAfter scaling (mean, std):")
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)
print(X_scaled_df.agg(["mean", "std"]).T.round(3))


---
## 7. Choosing K: The Elbow Method

Unlike classification, K-Means does not know the "correct" number of clusters -- we must choose **k**.

The **elbow method** trains K-Means for a range of k values and plots **inertia**
(the sum of squared distances from each point to its cluster centroid).
Inertia always decreases as k increases, but the rate of improvement slows down.
The "elbow" -- where the curve bends -- is a reasonable choice for k.


In [ ]:
inertias = []
k_range = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(list(k_range), inertias, marker="o", color="#2d6a9f", linewidth=2)
ax.set_xlabel("Number of Clusters (k)")
ax.set_ylabel("Inertia")
ax.set_title("Elbow Method for Choosing k")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

for k, inertia in zip(k_range, inertias):
    print(f"k={k}: inertia={inertia:.1f}")


> **Reading the plot:** look for the point where the curve stops dropping steeply and starts to flatten.
> We have **5 known formations** in this dataset (`Shale_Cap`, `Reservoir_1`, `Transition`, `Reservoir_2`, `Basement`),
> so we will use **k = 5** below to see how well unsupervised clusters recover that known geology.


---
## 8. Train K-Means and Inspect Clusters


In [ ]:
k = 5
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

df["Cluster"] = cluster_labels

print(f"Trained K-Means with k={k}")
print("\nRows per cluster:")
print(df["Cluster"].value_counts().sort_index())


In [ ]:
# Average (unscaled) log values per cluster -- this is how we interpret what each cluster represents
cluster_profile = df.groupby("Cluster")[feature_cols].mean().round(3)
cluster_profile


<div style="background: #f0f7ff; border-left: 5px solid #2d6a9f; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

### Reading the Cluster Profile Table

Each row above is a cluster's **average log response**. Use petrophysical intuition to interpret each one, e.g.:

- Low `GR_API`, low `SW_frac`, moderate `NPHI_frac` -&gt; likely a clean, hydrocarbon-bearing reservoir zone
- High `GR_API`, high `SW_frac` -&gt; likely a shale-dominated zone
- Very low `NPHI_frac`, high `RHOB_gcc` -&gt; likely a dense, low-porosity zone (possible basement)

This is the manual interpretation step that turns an anonymous cluster number into a geological zone name.

</div>


---
## 9. Compare Clusters to Known Formations

We happen to have `Formation` labels for this dataset, so we can check how well the unsupervised clusters
line up with the real geology. In a brand-new field, you would not have this comparison available --
this step is a sanity check on the method, not something clustering normally requires.


In [ ]:
# Cross-tabulate discovered clusters against known formations
contingency = pd.crosstab(df["Cluster"], df["Formation"])
contingency


In [ ]:
# Visualise the cross-tabulation as a heatmap
fig, ax = plt.subplots(figsize=(7.5, 5))
im = ax.imshow(contingency.values, cmap="Blues")

ax.set_xticks(range(len(contingency.columns)))
ax.set_yticks(range(len(contingency.index)))
ax.set_xticklabels(contingency.columns, rotation=35, ha="right")
ax.set_yticklabels([f"Cluster {c}" for c in contingency.index])
ax.set_xlabel("Known Formation")
ax.set_ylabel("K-Means Cluster")
ax.set_title("K-Means Clusters vs Known Formations")

for i in range(len(contingency.index)):
    for j in range(len(contingency.columns)):
        ax.text(j, i, contingency.values[i, j], ha="center", va="center", color="black")

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


<div style="background: #fff8e1; border-left: 5px solid #e0a800; padding: 14px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

### Interpreting the Match

- If most of a cluster's rows fall under one formation, K-Means has largely rediscovered that geology **without labels**.
- Mismatches are expected, especially at **transition zones**, where log responses genuinely blend between formations.
- Cluster numbers do not have to match formation order -- e.g. `Cluster 3` might correspond mostly to `Reservoir_1`. That is normal; only the *grouping*, not the *number*, is meaningful.

</div>


---
## 10. Dimensionality Reduction with PCA

We cannot easily plot 5 log features at once. PCA compresses them into 2 components for visualization.


In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

df["PC1"] = X_pca[:, 0]
df["PC2"] = X_pca[:, 1]

print("Explained variance ratio per component:")
for i, var in enumerate(pca.explained_variance_ratio_, start=1):
    print(f"  PC{i}: {var:.3f} ({var*100:.1f}% of variance)")
print(f"\nTotal variance captured by PC1 + PC2: {pca.explained_variance_ratio_.sum()*100:.1f}%")


In [ ]:
# Which original features contribute most to each component
loadings = pd.DataFrame(
    pca.components_.T,
    columns=["PC1", "PC2"],
    index=feature_cols
).round(3)
loadings


> A large positive or negative loading means that feature strongly influences that component.
> For example, if `GR_API` and `SW_frac` both load heavily on PC1, that axis likely separates shale-rich from clean, hydrocarbon-bearing rock.


---
## 11. Visualizing Clusters in PCA Space

We now plot every row in the 2D PCA space, once colored by the **unsupervised cluster** and once colored by the
**known Formation**, side by side.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Left: colored by K-Means cluster
scatter1 = axes[0].scatter(
    df["PC1"], df["PC2"], c=df["Cluster"], cmap="tab10", s=14, alpha=0.75
)
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].set_title("Colored by K-Means Cluster")
legend1 = axes[0].legend(*scatter1.legend_elements(), title="Cluster", loc="best", fontsize=8)
axes[0].add_artist(legend1)

# Right: colored by known Formation
formation_order = ["Shale_Cap", "Reservoir_1", "Transition", "Reservoir_2", "Basement"]
colors = ["#8e44ad", "#2ca87f", "#e0a800", "#2d6a9f", "#c0392b"]
for formation, clr in zip(formation_order, colors):
    subset = df[df["Formation"] == formation]
    axes[1].scatter(subset["PC1"], subset["PC2"], label=formation, color=clr, s=14, alpha=0.75)
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].set_title("Colored by Known Formation")
axes[1].legend(fontsize=8, loc="best")

fig.suptitle("PCA Projection: Unsupervised Clusters vs Known Geology", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


<div style="background: #f0f7ff; border-left: 5px solid #2d6a9f; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

### What to Look For

If the two panels show **similar shapes and groupings** (even with different colors), the unsupervised clustering
has captured real geological structure using only the log measurements -- no formation labels required.

This is the core value of unsupervised learning for reservoir zonation: it can propose zone boundaries in a
new well or field **before** any core-based or biostratigraphic labels exist.

</div>


---
## 12. Student Activity

<div style="background: #fff3cd; border-left: 5px solid #ffc107; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

### Activity: Try a Different k

Repeat Sections 8-9 with **k = 3** and then **k = 7**.

For each value of k:

1. Fit K-Means and produce the cluster profile table (Section 8)
2. Produce the cross-tabulation against `Formation` (Section 9)

Record your observations:

| k | Clusters vs 5 known formations | Comment |
|---|---|---|
| 3 | | |
| 5 (baseline) | | |
| 7 | | |

**Reflection:** Does a larger k always give a "better" zonation? What would you actually use to decide the right k
if you had no formation labels at all?

</div>


In [ ]:
# Activity -- try k=3 and k=7
# Hint: reuse the KMeans / groupby / crosstab code from Sections 8 and 9



---
## 13. Recap & Homework

<div style="background: #e8f5e9; border-left: 5px solid #2ca87f; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">

### What You Learned

| Concept | Meaning |
|---------|---------|
| Unsupervised learning | Finding structure in data with no target label |
| K-Means | Groups rows into k clusters based on feature similarity |
| Inertia / Elbow method | A heuristic for choosing a reasonable number of clusters |
| Feature scaling | Required before distance-based methods like K-Means and PCA |
| Cluster profile | Average feature values per cluster, used to interpret what a cluster represents |
| PCA | Compresses correlated features into a small number of components for visualization |
| Explained variance | How much of the original information a principal component retains |

Clustering does not replace geological interpretation -- it proposes structure that an engineer or geoscientist
must still confirm and name.

</div>


### Homework

Complete the following tasks in the code cells below.

**Task 1 -- Cluster Without Resistivity**

Repeat the K-Means workflow (Sections 6-8) using only `GR_API`, `NPHI_frac`, and `RHOB_gcc` (drop `RT_ohmm` and `SW_frac`).
Compare the new cross-tabulation against the original 5-feature result. Write two sentences on what changed and why.

**Task 2 -- Silhouette Score**

Look up `sklearn.metrics.silhouette_score`. Compute it for k = 2 through 8 on the scaled data and plot the result
alongside your elbow plot. Does the silhouette score agree with the k you chose visually from the elbow curve?

**Task 3 (Engineering) -- Zone a Single Well**

Filter the data to `Well_C` only, scale its features independently, and run K-Means with k = 4.
Plot `Depth_m` on the y-axis (inverted, so shallow is at the top) against cluster color to produce a simple
zoned log display for that one well. Comment on whether the zones look geologically continuous with depth.


In [ ]:
# Homework Task 1 -- Cluster Without Resistivity
# Hint: feature_cols_reduced = ["GR_API", "NPHI_frac", "RHOB_gcc"]



In [ ]:
# Homework Task 2 -- Silhouette Score
# Hint: from sklearn.metrics import silhouette_score



In [ ]:
# Homework Task 3 (Engineering) -- Zone a Single Well
# Hint: well_c = df[df["Well_ID"] == "Well_C"].sort_values("Depth_m")



<div style="background: #1a3a5c; color: white; padding: 22px 26px; border-radius: 10px; margin-top: 18px;">
  <h3 style="margin-top: 0; color: #ffffff;">Lesson Complete</h3>
  <p style="margin-bottom: 0; color: #d8ecff;">
    You have moved from predicting known labels to discovering structure with no labels at all.
    Clustering and PCA give you a first-pass reservoir zonation before any formation tops are picked --
    a genuinely useful tool for new wells and new fields.
    Next: <strong>Lesson 12 -- Model Optimization: Hyperparameter Tuning &amp; Explainable AI</strong>
  </p>
</div>
